# Double ResNet + CCIM — Train

| Dataset | Train | Val/Test |
|-------------|-------|----------|
| EMOTIC | EMOTIC 70% | EMOTIC 15%/15% |
| NCAERS | NCAERS train | NCAERS val/test |
| SynthCAER | Synth train | Synth val/test |

**Pipeline:** encoders → merge → joint_feature → cofounder dict (K-Means) → CCIM → logits.

## 1. Imports & config

In [ ]:
import json
from pathlib import Path

from dual_resnet_shared import (
    device,
    make_emotic_loaders, make_ncaers_loaders,
    make_synth_loaders, build_synth_index,
)
from ccim_model import (
    FinalModel, build_confounder_dictionary, train_single_label
)

CKPT_DIR = Path("checkpoints")
CKPT_DIR.mkdir(exist_ok=True)

print(f"Device: {device}")

## 2. EMOTIC

In [ ]:
# Loaders
emotic_train_loader, emotic_val_loader, emotic_test_loader, emotic_weights = make_emotic_loaders()

In [ ]:
emotic_model = FinalModel(freeze_backbone=True).to(device)

emotic_centers, emotic_prior = build_confounder_dictionary(
    emotic_model, emotic_train_loader,
    cache_path=CKPT_DIR / "emotic_confounders.pt",
)
emotic_model.set_confounders(emotic_centers.to(device), emotic_prior.to(device))

In [ ]:
emotic_history = train_single_label(
    emotic_model, emotic_train_loader, emotic_val_loader,
    emotic_weights, "EMOTIC", "EMOTIC",
    CKPT_DIR / "emotic_best.pt",
)

with open(CKPT_DIR / "emotic_history.json", "w") as f:
    json.dump(emotic_history, f)
print("History saved.")

## 3. NCAER-S

In [ ]:
ncaers_train_loader, ncaers_val_loader, ncaers_test_loader, ncaers_weights, ncaers_df = make_ncaers_loaders()

In [ ]:
ncaers_model = FinalModel(freeze_backbone=True).to(device)

ncaers_centers, ncaers_prior = build_confounder_dictionary(
    ncaers_model, ncaers_train_loader,
    cache_path=CKPT_DIR / "ncaers_confounders.pt",
)
ncaers_model.set_confounders(ncaers_centers.to(device), ncaers_prior.to(device))

In [ ]:
ncaers_history = train_single_label(
    ncaers_model, ncaers_train_loader, ncaers_val_loader,
    ncaers_weights, "CAER_S", "NCAERS",
    CKPT_DIR / "ncaers_best.pt",
)

with open(CKPT_DIR / "ncaers_history.json", "w") as f:
    json.dump(ncaers_history, f)
print("History saved.")

## 4. SynthCAER

In [ ]:
synth_df = build_synth_index()
synth_train_loader, synth_val_loader, synth_test_loader, synth_weights = make_synth_loaders(synth_df)

In [ ]:
synth_model = FinalModel(freeze_backbone=True).to(device)

synth_centers, synth_prior = build_confounder_dictionary(
    synth_model, synth_train_loader,
    cache_path=CKPT_DIR / "synth_confounders.pt",
)
synth_model.set_confounders(synth_centers.to(device), synth_prior.to(device))

In [ ]:
synth_history = train_single_label(
    synth_model, synth_train_loader, synth_val_loader,
    synth_weights, "SynthCAER", "SynthCAER",
    CKPT_DIR / "synth_best.pt",
)

with open(CKPT_DIR / "synth_history.json", "w") as f:
    json.dump(synth_history, f)
print("History saved.")